# Lesson 37 · The brute-force DFT — interrogating a signal one frequency at a time: an O(N²) double loop that computes your first spectrum by hand

**Goal**: implement the $O(N^2)$ naive DFT and understand that "each frequency bin = the dot product of the signal with one column of twiddle factors".

> **A heart-of-the-course lesson, clarity first**: the double loop is allowed to be slow, but you should compute your first spectrum yourself. Frequency bin $k$ asks: "how much of the signal turns at this speed?"

🔗 **Aurora connection**: `dft()` in `transforms.py` is the reference answer; once you've written yours, check it against the reference / `numpy.fft` with `np.allclose`.

## Review bridge · the DFT matrix from L21 (2 minutes)

L21 already viewed the DFT as one matrix multiplication:

```text
X = W @ x
```

where each row of `W` is a frequency basis vector. L37 no longer starts from the matrix view;
instead it unpacks the same thing into loops:

```text
X[k] = Σ x[n] · e^{-2πikn/N}
```

The two forms are equivalent:

- The matrix version helps you remember: the DFT is a linear transform
- The loop version helps you remember: each `X[k]` is one "signal vs frequency basis" dot product
- L38–L39 will go on to ask: which repeated computations inside these dot products can we skip?

If the formula suddenly feels steep, look back at L21's `W @ x` first, then come back and read this lesson's `for k in range(N)`.


In [ ]:
# Aurora matplotlib bootstrap
from pathlib import Path
import sys

_root = None
_cwd = Path.cwd().resolve()
for _candidate in (_cwd, *_cwd.parents):
    if (_candidate / '_matplotlib_bootstrap.py').exists():
        _root = _candidate
        break
if _root is None:
    _notebooks_dir = _cwd / 'notebooks'
    if _notebooks_dir.exists():
        for _found in _notebooks_dir.rglob('_matplotlib_bootstrap.py'):
            _root = _found.parent
            break
if _root is not None and str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from _matplotlib_bootstrap import apply as _aurora_mpl_apply
_aurora_mpl_apply()


In [ ]:
import numpy as np

N = 4
x = np.array([1.0, 0.0, -1.0, 0.0])
k = np.arange(N).reshape(N, 1)
n = np.arange(N).reshape(1, N)
W = np.exp(-2j * np.pi * k * n / N)

X_matrix = W @ x
X_numpy = np.fft.fft(x)

print("W @ x       =", np.round(X_matrix, 4))
print("np.fft.fft =", np.round(X_numpy, 4))
np.testing.assert_allclose(X_matrix, X_numpy, atol=1e-12)
print("✅ The L21 matrix view and the L37 formula view are exactly equivalent")


← **Previous lesson** · [L36 · Window functions](L36_windows.ipynb)

> Last lesson covered **window functions**: spectral leakage from the rectangular window; Hann / Hamming / Blackman compared.
> This lesson explores **the brute-force DFT**.

## This lesson's story: why doesn't spectral analysis just "sweep the frequencies"?

Imagine you want to know which notes are in a piece of music. One brute-force approach: for every possible frequency (say 0 to 20000 Hz), ask separately "how much of this frequency is in the signal?" — scanning them one by one.

That's exactly the principle of the **DFT (discrete Fourier transform)**, and what people did before the FFT:

$$X[k] = \sum_{n=0}^{N-1} x[n] \cdot e^{-2\pi i k n / N}$$

How to read it: for the `k`-th frequency, use the twiddle factor `e^{-2πikn/N}` to "question" the signal `x[n]` point by point, then sum — this is the **dot product** of the signal with the k-th frequency basis vector.

**A 4-point DFT by hand** (`x = [1, 0, -1, 0]`, a cosine wave at k=1):

```
X[0] = 1 + 0 + (-1) + 0 = 0          ← the DC component is zero
X[1] = 1·e⁰ + 0·e^{-πi/2} + (-1)·e^{-πi} + 0·e^{-3πi/2}
     = 1 + 0 + 1 + 0 = 2             ← there is signal at k=1 ✓
X[2] = 1·e⁰ + 0·e^{-πi} + (-1)·e^{-2πi} + 0·e^{-3πi}
     = 1 + 0 - 1 + 0 = 0
X[3] = 1·e⁰ + 0·e^{-3πi/2} + (-1)·e^{-3πi} + 0·e^{-9πi/2}
     = 1 + 0 + 1 + 0 = 2              ← e^{-3πi} = -1, so (-1)·(-1) = +1
```

Conclusion: `X[1]=2` correctly reflects that x[n] is a cosine at k=1; `X[3]=2` is its mirror image — a real signal satisfies conjugate symmetry `X[3] = conj(X[1]) = 2` (section 3 covers this properly). The whole spectrum is `[0, 2, 0, 2]`, matching `np.fft.fft([1,0,-1,0])`.

**Time complexity**: N frequency bins × N-term dot products = O(N²). Double N and the time goes up 4×; the FFT brings it down to O(N log N) with butterfly divide-and-conquer — that's L38's topic.

This section's task: implement `naive_dft(x)` with a double loop and verify it point by point against `numpy.fft.fft`.

### Pause: how do you work out the complex values in the hand calculation?

In the hand calculation above we used e^{-πi/2} = -i, e^{-πi} = -1, e^{-3πi/2} = i. How do you get these step by step?

**Euler's formula** is the key: for any real θ,
$$e^{i\theta} = \cos(\theta) + i \sin(\theta)$$

So for a negative angle, just negate the sine term:
$$e^{-i\theta} = \cos(\theta) - i \sin(\theta)$$

**Working out e^{-πi} (it appears in the X[1] example)**:
- θ = π, so e^{-πi} = cos(π) - i·sin(π) = (-1) - i·(0) = **-1**  ✓

**Working out e^{-πi/2} (also in the X[1] example)**:
- θ = π/2, so e^{-πi/2} = cos(π/2) - i·sin(π/2) = 0 - i·(1) = **-i**  ✓

**Working out e^{-3πi/2} (the 4th term of X[1])**:
- θ = 3π/2 = 270°, so e^{-3πi/2} = cos(3π/2) - i·sin(3π/2) = 0 - i·(-1) = **i**  ✓

**Pattern table** (twiddle factors that come up often in an N=4 DFT):

| angle θ | e^{iθ} | geometric meaning |
|---|---|---|
| 0 | 1 | right side of the unit circle |
| π/2 (90°) | i | top of the unit circle |
| π (180°) | -1 | left side of the unit circle |
| 3π/2 (270°) | -i | bottom of the unit circle |
| -π/2 (-90°) | -i | bottom of the unit circle (clockwise) |
| -π (-180°) | -1 | left side of the unit circle (clockwise) |

So in the hand calculation above:
- x[2]=-1 times e^{-πi}=-1 → (-1)×(-1) = +1  ✓
- x[3]=0 times any twiddle factor → 0

The full hand check: X[1] = 1·(1) + 0·(-i) + (-1)·(-1) + 0·(i) = 1 + 0 + 1 + 0 = 2  ✓

In [ ]:
import numpy as np
import time

## Orientation map: the whole N=8 DFT matrix

Look at this heat map before deriving the formula. Each row is a **cosine/sine basis vector**; `X[k]` is the dot product of the signal `x` with row `k`. The matrix takes N² multiplications in all — exactly where O(N²) comes from, and the redundancy the FFT sets out to eliminate.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

N = 8
k, n = np.mgrid[0:N, 0:N]
F = np.exp(-2j * np.pi * k * n / N)

fig, axes = plt.subplots(1, 2, figsize=(9, 3.8))
for ax, data, title, cmap in zip(
    axes, [F.real, F.imag],
    ['Re(F)  cosine basis', 'Im(F)  negative sine basis'],
    ['RdBu_r', 'PuOr_r'],
):
    im = ax.imshow(data, cmap=cmap, vmin=-1, vmax=1)
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.set_xlabel('sample n'); ax.set_ylabel('frequency bin k')
    ax.set_xticks(range(N)); ax.set_yticks(range(N))
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

fig.suptitle('N=8 DFT matrix  F[k,n] = exp(−2πi·k·n/8)\n'
             'dot product of row k with signal x = X[k] (strength and phase of bin k)',
             fontsize=10.5, fontweight='bold')
plt.tight_layout()
plt.show()

### Prerequisite: the dot product of complex vectors, and "perpendicular"

Before talking about the unitarity of the DFT matrix, we need to clear something up: **how does the dot product of complex vectors differ from that of real vectors?**

The real dot product is familiar:
$$\langle \mathbf{u}, \mathbf{v} \rangle = u_0 v_0 + u_1 v_1 + \cdots + u_{N-1} v_{N-1}$$

The complex dot product has one extra detail: **the first vector is conjugated**. This guarantees that the "squared length" from an inner product is always real and non-negative:
$$\langle \mathbf{u}, \mathbf{v} \rangle = u_0^* v_0 + u_1^* v_1 + \cdots + u_{N-1}^* v_{N-1}$$

where $u^* = a - bi$ (if u = a + bi) is the complex conjugate.

**Example**: compute the dot product of $\mathbf{u} = [1+i, 2]$ and $\mathbf{v} = [1, 1-i]$:
$$\langle \mathbf{u}, \mathbf{v} \rangle = (1+i)^* \cdot 1 + 2^* \cdot (1-i) = (1-i) \cdot 1 + 2 \cdot (1-i) = (1-i) + (2-2i) = 3-3i$$

**The definition of "perpendicular" for complex vectors**: the dot product is 0, i.e. $\langle \mathbf{u}, \mathbf{v} \rangle = 0$.

This definition matters because, in the DFT matrix, different rows (basis vectors of different frequencies) are mutually perpendicular under it.

## 1. The DFT formula and its matrix form

The formula for a single frequency bin:

```
X[k] = sum_{n=0}^{N-1} x[n] * exp(-2*pi*i*k*n/N)
```

Vectorized: let `w_k = [exp(-2*pi*i*k*n/N) for n in 0..N-1]`; then `X[k] = dot(w_k, x)`.

Note that this "dot" is the plain sum `dot(w_k, x) = sum_n w_k[n] * x[n]`, with no extra conjugate — the conjugation is already built into the minus sign of `w_k`. In complex inner-product language (first vector conjugated), the same thing reads `X[k] = ⟨e_k, x⟩ = sum_n conj(e_k[n]) * x[n]` with `e_k[n] = exp(+2*pi*i*k*n/N)`.

Stack all the rows k=0..N-1 and you get the **DFT matrix** `F`, with `X = F @ x`.

### Unitary matrices and complex linear algebra

`F` is a special kind of matrix: a **unitary matrix**. A unitary matrix is the complex version of an orthogonal matrix.

Definition: a matrix $U$ is unitary if and only if
$$U^H U = I$$

where $U^H$ (read "U-Hermitian") is the conjugate transpose of U: transpose first, then conjugate every element.

**Unitarity of the DFT matrix** (not quite exact, see below): the raw DFT matrix F does not itself satisfy $FF^H = I$, but **it does after dividing by $\sqrt{N}$**:
$$F_{\text{norm}} = \frac{1}{\sqrt{N}} F, \quad F_{\text{norm}}^H F_{\text{norm}} = I$$

This means:
1. **Basis vectors of different frequencies are mutually perpendicular** (under the complex inner product)
2. **Energy is conserved**: the signal's total power = the spectrum's total power (Parseval's theorem)
3. **Invertibility**: the IDFT (inverse transform) exists and has an explicit formula

In practice we usually **don't** divide by $\sqrt{N}$ (to keep things simple), but remember this point for theoretical analysis or energy calculations.

In [ ]:
# Demo: build the 4×4 DFT matrix and verify its unitarity
N = 4
k = np.arange(N).reshape(N, 1)   # column vector
n = np.arange(N).reshape(1, N)   # row vector
F = np.exp(-2j * np.pi * k * n / N)
F_norm = F / np.sqrt(N)

# verify unitarity: F_norm @ F_norm^H ≈ I
residual = np.abs(F_norm @ F_norm.conj().T - np.eye(N)).max()
print(f"DFT matrix (N=4):")
print(np.round(F, 2))
print(f"\nunitarity residual (should be ≈0): {residual:.2e}")

## 2. The twiddle factor W_N and the physical meaning of the minus sign

Define the basic twiddle factor:

```
W_N = exp(-2*pi*i / N)
```

The matrix elements can be written as:

```
F[k, n] = W_N^{k*n}
```

`W_N^{k*n}` is the point on the unit circle that has **turned k*n/N of a full turn clockwise** (in the complex plane, a negative angle means clockwise rotation). As k grows, each frequency basis vector turns faster, corresponding to a higher-frequency oscillation. `W_N` is an N-th root of unity, satisfying `W_N^N = 1` — the root of the symmetry that the FFT butterfly exploits.

### Why a minus sign rather than a plus sign?

The choice isn't arbitrary. **Physically**, the DFT is defined with a minus sign because:
- In signal processing, we "correlate" the signal against the complex exponential $e^{i \omega t}$ (counterclockwise rotation, a positive frequency)
- The minus sign amounts to **rotating the opposite way** (clockwise), so that when the signal really contains an $e^{i \omega_k t}$ component, multiplying it by $e^{-i \omega_k t}$ gives a DC component (the two rotations cancel) — which is exactly why the DFT can "detect" that frequency component
- This is also the "analysis direction" that L35 (Euler meets the FFT) talked about

**An intuitive analogy**: imagine someone drumming at a fixed rate, and you want to detect whether their rate equals some target rate. The best way is to drum at that same rate yourself (in the opposite sense): if the rates match, the two rhythms cancel into a steady signal (large energy); if they differ, they keep sliding against each other (small energy). That's why the minus sign is used.

In [ ]:
# Demo: where the N=8 twiddle factors sit on the unit circle
N = 8
W = np.exp(-2j * np.pi / N)
roots = [W**k for k in range(N)]
print(f"twiddle factors for N={N} (k=0..{N-1}):")
for k, w in enumerate(roots):
    print(f"  W_8^{k} = {w.real:+.4f} {w.imag:+.4f}i  |magnitude|={abs(w):.4f}")
print(f"\nW_N^N = {W**N:.6f} (should be 1+0i)")

## 3. What frequencies mean, and conjugate symmetry

### The physical frequency, and deriving its formula

The physical frequency (Hz) that spectrum bin `X[k]` corresponds to:

```
f_k = k * sr / N
```

where `sr` is the sample rate.

**Why this formula?** Let's derive it from the link between sampling and frequency:
- Sample rate sr, N samples → duration T = N / sr (how long the signal is in total)
- The fundamental frequency of the time window (lowest non-zero frequency) = 1 / T = sr / N
- The k-th frequency bin = k × (fundamental) = k × sr / N  ✓

**Example**: sr=8000 Hz, N=8
- f_0 = 0 Hz (DC)
- f_1 = 1000 Hz (lowest useful frequency)
- f_2 = 2000 Hz
- f_3 = 3000 Hz
- f_4 = 4000 Hz (Nyquist frequency = sr/2, the highest distinguishable frequency)

### Conjugate symmetry: why does a real signal satisfy X[k] = conj(X[N-k])?

For a **real signal** `x[n]`, the DFT necessarily has conjugate symmetry. Here's the derivation:

Let x be a real sequence, and compute X[N-k]:
$$X[N-k] = \sum_{n=0}^{N-1} x[n] \cdot e^{-2\pi i(N-k)n/N}$$

Expand the exponential:
$$e^{-2\pi i(N-k)n/N} = e^{-2\pi i N n/N} \cdot e^{2\pi i k n/N} = e^{-2\pi i n} \cdot e^{2\pi i k n/N} = 1 \cdot e^{2\pi i k n/N} = e^{2\pi i k n/N}$$

So:
$$X[N-k] = \sum_{n=0}^{N-1} x[n] \cdot e^{2\pi i k n/N}$$

On the other hand, compute $\text{conj}(X[k])$:
$$\text{conj}(X[k]) = \text{conj}\left(\sum_{n=0}^{N-1} x[n] \cdot e^{-2\pi i k n/N}\right)$$

Because x[n] is real, conjugation only affects the complex exponential:
$$\text{conj}(X[k]) = \sum_{n=0}^{N-1} x[n] \cdot \text{conj}(e^{-2\pi i k n/N}) = \sum_{n=0}^{N-1} x[n] \cdot e^{2\pi i k n/N}$$

Comparing the two, we get:
$$X[N-k] = \text{conj}(X[k])$$  ✓

**Physical meaning**: this symmetry means that the spectral information of a real signal is **already complete in the first N/2+1 bins**; the later bins mirror the earlier ones and carry no new information. This is the basis of the RFFT (real FFT) optimization.

### What negative frequencies mean

By index k, the DFT output falls into three parts:
- **k=0**: the DC component
- **k=1...N//2**: positive frequencies (0 Hz up to the Nyquist frequency)
- **k=N//2+1...N-1**: these correspond to **negative frequencies** (-sr/2 up to 0 Hz)

The term "negative frequency" comes from complex-signal theory: the complex exponential $e^{-2\pi i f t}$ has frequency -f (clockwise rotation). But for a real signal, this "negative frequency" part is just the conjugate mirror of the positive-frequency part and contains no extra information.

Think of it this way: **a real signal can be seen as the sum of a complex signal's positive and negative frequencies, symmetrically paired**. A real cosine $\cos(2\pi f t)$ splits into two complex exponentials:
$$\cos(2\pi f t) = \frac{1}{2}(e^{2\pi i f t} + e^{-2\pi i f t})$$

So in the DFT, the "positive frequency" at $k$ and the "negative frequency" at $N-k$ must be conjugate-symmetric.

In [ ]:
# Demo: the DC component + conjugate symmetry
x = np.array([1.0, 2.0, 3.0, 4.0])
X = np.fft.fft(x)
N = len(x)
print(f"x = {x}")
print(f"X = {np.round(X, 4)}")
print(f"\nX[0] (DC) = {X[0].real:.1f}, sum(x) = {sum(x):.1f}")
print(f"\nconjugate-symmetry check: X[k] == conj(X[N-k])")
for k in range(1, N):
    match = np.isclose(X[k], np.conj(X[N-k]))
    print(f"  X[{k}] vs conj(X[{N-k}]): {match}")

In [ ]:
### Check: the complex dot product and conjugate symmetry

# Demo 1: the complex dot product (with conjugation)
import numpy as np

u = np.array([1+1j, 2])  # [1+i, 2]
v = np.array([1, 1-1j])  # [1, 1-i]

# the correct complex dot product: conjugate the first vector
dot_product = np.sum(np.conj(u) * v)
print(f"u = {u}")
print(f"v = {v}")
print(f"⟨u, v⟩ = sum(conj(u)*v) = {dot_product}")

# Demo 2: conjugate symmetry of a real signal
x_real = np.array([1.0, 2.0, 3.0, 4.0])
X_real = np.fft.fft(x_real)
N = len(x_real)

print(f"\nreal signal x = {x_real}")
print(f"X = {np.round(X_real, 4)}")
print(f"\nconjugate-symmetry check: X[k] ?= conj(X[N-k])")
for k in range(1, N):
    Xk = X_real[k]
    XNk_conj = np.conj(X_real[N-k])
    match = np.isclose(Xk, XNk_conj)
    print(f"  k={k}: X[{k}]={Xk:.4f}  vs  conj(X[{N-k}])={XNk_conj:.4f}  {match}")

# Demo 3: checking the hand calculation
print("\nchecking the hand calculation (X[1] and X[3]):")
x = np.array([1.0, 0.0, -1.0, 0.0])
N = 4
# X[1] by hand
k = 1
w_1 = np.exp(-2j * np.pi * k * np.arange(N) / N)  # sequence of twiddle factors
X_1_manual = np.sum(x * w_1)  # vector dot product
X_1_numpy = np.fft.fft(x)[1]
print(f"X[1] by hand: {np.round(X_1_manual, 4)}, numpy: {np.round(X_1_numpy, 4)}")
print(f"twiddle factors w_1 = {np.round(w_1, 4)}")
print(f"element-wise products = {np.round(x * w_1, 4)}, sum = {np.round(X_1_manual, 4)}")

# X[3] by hand
k = 3
w_3 = np.exp(-2j * np.pi * k * np.arange(N) / N)
X_3_manual = np.sum(x * w_3)
X_3_numpy = np.fft.fft(x)[3]
print(f"\nX[3] by hand: {np.round(X_3_manual, 4)}, numpy: {np.round(X_3_numpy, 4)}")
print(f"twiddle factors w_3 = {np.round(w_3, 4)}")
print(f"element-wise products = {np.round(x * w_3, 4)}, sum = {np.round(X_3_manual, 4)}")

print(f"\nX[1] = conj(X[3])? {np.isclose(X_1_numpy, np.conj(X_3_numpy))}")
print(f"since x is real, X[3] = conj(X[4-3]) = conj(X[1]), so X[3] is the conjugate of X[1]")

## 4. ✏️ Implement `naive_dft(x)`

**A three-step implementation route**:

| Step | Action | Code skeleton |
|---|---|---|
| 1 | Preallocate the output | `X = np.zeros(N, dtype=complex)` |
| 2 | Outer loop over k (N frequency bins) | `for k in range(N):` |
| 3 | Vectorize the inner part (N time samples) | `n = np.arange(N); X[k] = np.dot(x, np.exp(-2j*np.pi*k*n/N))` |

**Key points**:
- **Complex dot product**: mathematically the first vector is conjugated; in NumPy, only `np.vdot(w_k, x)` conjugates its first argument automatically. If you write the twiddle factor straight from the formula as `np.exp(-2j*np.pi*k*n/N)` and then use `np.dot(x, ...)`, you're computing exactly the sum in the DFT.
- **Twiddle factor**: `np.exp(-2j*np.pi*k*n/N)` gives the complex value of $e^{-2\pi i k n/N}$
- **Acceptance criterion**: `np.allclose(naive_dft(x), np.fft.fft(x))` passes (atol=1e-10)

**Common mistakes**:
- Forgetting the minus sign: `exp(+2j*...)` is the inverse DFT, not the DFT
- Real input: you first need `x = np.asarray(x, dtype=complex)` (the stub already does this)
- A double loop without vectorizing works too: loop over n with `X[k] += x[n] * np.exp(...)`

**DFT physical frequency**: the corresponding physical frequency is `f_k = k × sr / N` Hz, so k=1 is the lowest non-zero frequency, `sr/N` Hz.

In [ ]:
def naive_dft(x: np.ndarray) -> np.ndarray:
    """Naive O(N²) DFT: X[k] = sum_n x[n]*exp(-2*pi*i*k*n/N)"""
    x = np.asarray(x, dtype=complex)
    N = len(x)
    X = np.zeros(N, dtype=complex)
    n = np.arange(N)
    # delete the raise line below, then fill in your implementation in the for loop
    raise NotImplementedError("Implement naive_dft: following the DFT formula, compute the twiddle factors for each k and take the dot product")
    for k in range(N):
        # ✏️ TODO: from the DFT definition, compute the N twiddle factors for frequency index k (a complex sequence)
        # ✏️ TODO: take the dot product of the signal x with the twiddle-factor sequence and store it in X[k]
        pass
    return X


In [ ]:
# Check: compare against numpy's FFT
x = np.array([1.0, 2.0, 3.0, 4.0])
try:
    _out = naive_dft(x)
    _not_impl = False
except (NotImplementedError, TypeError):
    _out = None
    _not_impl = True

if _not_impl or _out is None or not np.any(_out):
    print("⬜ Finish the TODO items in naive_dft first, then run this cell")
else:
    np.testing.assert_allclose(_out, np.fft.fft(x), atol=1e-10)
    print("✅ naive_dft([1,2,3,4]) matches np.fft.fft with error < 1e-10")
    for N in [8, 16, 32]:
        xr = np.random.randn(N)
        np.testing.assert_allclose(naive_dft(xr), np.fft.fft(xr), atol=1e-10)
    print(f"✅ random inputs N=8,16,32 all pass")


## 5. Parameter experiment: verifying O(N²) time complexity

Time `naive_dft` for `N = 8, 64, 256`:
- N grows 8× → time is expected to grow **64×** (the N² law)
- Once the FFT is implemented (L38–L39), N=256 drops from ~milliseconds to ~microseconds (about a **1000× speedup**)

In [ ]:
# In-depth demo: the complete 4-point DFT by hand
print("=" * 60)
print("Complete 4-point DFT, worked by hand")
print("=" * 60)

x = np.array([1.0, 0.0, -1.0, 0.0])
N = 4

print(f"\ninput signal: x = {x}")
print(f"this is a cosine wave at k=1: x[n] = cos(2π·1·n/4) = cos(πn/2)")
print(f"that is: x = [cos(0), cos(π/2), cos(π), cos(3π/2)] = [1, 0, -1, 0] ✓\n")

print("=" * 60)
print("computing bin k=0 (the DC component)")
print("=" * 60)
k = 0
angles = -2 * np.pi * k * np.arange(N) / N
w_k = np.exp(1j * angles)
print(f"twiddle-factor angles: -2π·0·n/4 = [0, 0, 0, 0] (radians)")
print(f"twiddle factors (points on the unit circle): e^{{j·0}} = {w_k}")
print(f"element-wise products: x * w_k = {x} * {w_k} = {x * w_k}")
X_k = np.sum(x * w_k)
print(f"sum: X[0] = {X_k}")
print(f"check: numpy.fft.fft(x)[0] = {np.fft.fft(x)[0]}\n")

print("=" * 60)
print("computing bin k=1 (the first useful frequency)")
print("=" * 60)
k = 1
angles = -2 * np.pi * k * np.arange(N) / N
print(f"twiddle-factor angles: -2π·1·n/4 = {np.round(angles, 4)} (radians)")
print(f"that is: -2π·1·[0,1,2,3]/4 = [0, -π/2, -π, -3π/2]")
w_k = np.exp(1j * angles)
print(f"twiddle factors (expanded with Euler's formula):")
for n in range(N):
    angle = angles[n]
    real_part = np.cos(angle)
    imag_part = np.sin(angle)
    print(f"  e^{{j·{angle:.4f}}} = e^{{j·{angle/np.pi:.2f}π}} = cos({angle:.4f}) + j·sin({angle:.4f}) = {real_part:.4f} + {imag_part:.4f}j")
print(f"twiddle-factor vector: w_1 = {np.round(w_k, 4)}")
products = x * w_k
print(f"element-wise products: x * w_1 =")
for n in range(N):
    print(f"  x[{n}]·w_1[{n}] = {x[n]:5.1f} · {np.round(w_k[n], 4)} = {np.round(products[n], 4)}")
X_k = np.sum(products)
print(f"sum: X[1] = {np.round(X_k, 4)}")
print(f"check: numpy.fft.fft(x)[1] = {np.round(np.fft.fft(x)[1], 4)}\n")

print("=" * 60)
print("computing bin k=2")
print("=" * 60)
k = 2
angles = -2 * np.pi * k * np.arange(N) / N
w_k = np.exp(1j * angles)
print(f"twiddle-factor angles: -2π·2·n/4 = [0, -π, -2π, -3π]")
print(f"since e^{{j·2π}} = 1, this simplifies to: [0, -π, 0, -π]")
print(f"twiddle-factor vector: w_2 = {np.round(w_k, 4)}")
products = x * w_k
print(f"element-wise products:")
for n in range(N):
    print(f"  x[{n}] · w_2[{n}] = {x[n]:5.1f} · {np.round(w_k[n], 4)} = {np.round(products[n], 4)}")
X_k = np.sum(products)
print(f"sum: X[2] = {np.round(X_k, 4)}")
print(f"check: numpy.fft.fft(x)[2] = {np.round(np.fft.fft(x)[2], 4)}\n")

print("=" * 60)
print("computing bin k=3 (a conjugate-symmetry example)")
print("=" * 60)
k = 3
angles = -2 * np.pi * k * np.arange(N) / N
w_k = np.exp(1j * angles)
print(f"twiddle-factor angles: -2π·3·n/4 = [0, -3π/2, -3π, -9π/2]")
print(f"twiddle-factor vector: w_3 = {np.round(w_k, 4)}")
products = x * w_k
X_k = np.sum(products)
print(f"element-wise products: {np.round(products, 4)}")
print(f"sum: X[3] = {np.round(X_k, 4)}")
X_1 = np.fft.fft(x)[1]
print(f"check: numpy.fft.fft(x)[3] = {np.round(np.fft.fft(x)[3], 4)}")
print(f"conjugate-symmetry check: conj(X[1]) = conj({np.round(X_1, 4)}) = {np.round(np.conj(X_1), 4)} ✓\n")

print("=" * 60)
print("final spectrum")
print("=" * 60)
X = np.fft.fft(x)
print(f"X = {np.round(X, 4)}")
print(f"magnitude spectrum: |X| = {np.round(np.abs(X), 4)}")
print(f"phase spectrum: ∠X = {np.round(np.angle(X), 4)} (radians)")
print("=" * 60)

In [ ]:
# Safety guard: skip the timing while the stub is unimplemented, to avoid meaningless output
_impl_ok = True
try:
    _timing_guard = naive_dft(np.array([1.0+0j, 0.0+0j]))
    if not np.any(_timing_guard):
        _impl_ok = False
except (NotImplementedError, TypeError):
    _impl_ok = False

if not _impl_ok:
    print("⬜ Implement naive_dft first, then run the timing cell")
else:
    print("=" * 70)
    print("Verifying O(N²) time complexity: the naive DFT implementation")
    print("=" * 70)
    results = {}
    
    for N in [8, 64, 256]:
        xr = np.random.randn(N).astype(complex)
        # timing: repeat 20 times and average
        times = []
        for trial in range(20):
            t0 = time.perf_counter()
            naive_dft(xr)
            t1 = time.perf_counter()
            times.append(t1 - t0)
        avg_time_ms = np.mean(times) * 1000  # convert to milliseconds
        results[N] = avg_time_ms
        print(f"N={N:4d}: avg time = {avg_time_ms:.6f} ms (20 trials, std={np.std(times)*1000:.6f} ms)")

    print("\n" + "=" * 70)
    print("Complexity analysis:")
    print("=" * 70)
    
    # verify N² scaling
    ratio_64_8   = results[64]  / results[8]
    ratio_256_64 = results[256] / results[64]
    ratio_256_8  = results[256] / results[8]
    
    print(f"time ratio N=64/N=8   = {ratio_64_8:6.1f}x  (theory 64x, error {abs(64-ratio_64_8)/64*100:.1f}%)")
    print(f"time ratio N=256/N=64  = {ratio_256_64:6.1f}x  (theory 16x, error {abs(16-ratio_256_64)/16*100:.1f}%)")
    print(f"time ratio N=256/N=8   = {ratio_256_8:6.1f}x   (theory 1024x, error {abs(1024-ratio_256_8)/1024*100:.1f}%)")
    
    print(f"\n✅ O(N²) complexity confirmed: time is proportional to N²")
    print(f"📊 If N grows from 256 to 1024 (×4), the run time grows by about 16×")

---
→ **Main-line order**: after finishing `naive_dft`, go on to [L38 · The FFT butterfly](L38_fft_butterfly.ipynb) first, then to [L39 · Writing the FFT from scratch](https://github.com/djzoom/AURORA/blob/main/notebooks/5_audio_dsp/L39_fft_implement.ipynb) *(Chinese)*.

→ **Come back later**: after finishing L39, open [L42 · The FFT drawn](https://github.com/djzoom/AURORA/blob/main/notebooks/5_audio_dsp/L42_visual_fft.ipynb) *(Chinese)* and test your intuition with the butterfly diagram and the spectral shapes of a pure tone, a chord and noise.


## Wrap-up

`naive_dft(x)` maps a signal of length N to N complex spectral coefficients `X[k]`; each output component gives the signal's magnitude and phase at frequency `k*sr/N` Hz. This implementation corresponds directly to the `dft()` function in Aurora's `src/aurora/audio/transforms.py`, and it is the algorithmic foundation for L38's FFT (the butterfly), L39's full FFT implementation, and L43's STFT (short-time sliding windows).

The next lesson (L38) speeds up the DFT's O(N²) double loop into O(N log N) butterfly divide-and-conquer — the core idea of the FFT.

---
⬇️ **Final check**: you've read the wrap-up; now complete the whiteboard challenge below before ticking the self-review.


## ✏️ Whiteboard challenge: the DFT by hand (target: 10 minutes)

Cover the screen and answer on paper:

**Question 1**: For `x = [1, 0, -1, 0]`, work out X[0] and X[2] by hand:
- X[0] = x[0]·e^0 + x[1]·e^0 + x[2]·e^0 + x[3]·e^0 = ? (the DC component)
- X[2] = x[0]·e^0 + x[1]·e^{-πi} + x[2]·e^{-2πi} + x[3]·e^{-3πi} = ?

**Question 2**: For `x = [1, 1, 1, 1]` (a DC signal), what should every X[k] be?
- X[0] = ?, X[1] = ?, X[2] = ?, X[3] = ?

**Question 3**: With N=8 and sr=8000 Hz, what physical frequency in Hz does k=3 correspond to?
(Formula: f_k = k × sr / N)

**Question 4**: naive_dft has time complexity O(N²). If N grows from 256 to 1024 (×4), roughly how many times longer does it take?

**Question 5**: X[N-k] = X[k]* (conjugate symmetry) — what does this mean for a real signal?
(Hint: for N=8 you only need to compute X[0]…X[4]; X[5]…X[7] come for free)

When you've finished the derivation, run the cell below to check your answers.

In [ ]:
# ✏️ Answer-check cell
import numpy as np

# Q1: x=[1,0,-1,0]
x = np.array([1.0, 0.0, -1.0, 0.0])
X_np = np.fft.fft(x)
assert np.isclose(X_np[0], 0+0j, atol=1e-12), f"X[0] should be 0, got {X_np[0]}"
assert np.isclose(X_np[2], 0+0j, atol=1e-12), f"X[2] should be 0, got {X_np[2]}"
try:
    X_mine = naive_dft(x)
    assert np.allclose(X_mine, X_np, atol=1e-10)
    print(f"Q1 ✅  X[0]={X_mine[0]:.4f}, X[2]={X_mine[2]:.4f} (naive_dft agrees with numpy)")
except (NotImplementedError, TypeError):
    print("⬜ Q1: implement naive_dft() first, then run this answer-check cell")

# Q2: x=[1,1,1,1], a DC signal → X[0]=4, the rest = 0
x_dc = np.array([1.0, 1.0, 1.0, 1.0])
X_dc = np.fft.fft(x_dc)
assert np.isclose(X_dc[0], 4+0j, atol=1e-12)
assert np.allclose(X_dc[1:], 0+0j, atol=1e-12)
print(f"Q2 ✅  DC signal: X[0]={X_dc[0]:.4f}, X[1:]=≈0 (all the energy is at k=0)")

# Q3: f_k = k*sr/N
k, sr, N = 3, 8000, 8
f_k = k * sr / N
assert f_k == 3000.0
print(f"Q3 ✅  k=3, sr=8000, N=8 → f_3 = {f_k:.0f} Hz")

# Q4: O(N²) time complexity — N×4 → time×16
ratio = (1024**2) / (256**2)
assert ratio == 16.0
print(f"Q4 ✅  N: 256→1024 (×4), time ≈ {ratio:.0f}× (O(N²))")

# Q5: checking conjugate symmetry
x_real = np.array([1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0])
X_real = np.fft.fft(x_real)
N8 = len(x_real)
for k_val in range(1, N8//2):
    assert np.isclose(X_real[k_val], np.conj(X_real[N8-k_val]), atol=1e-10)
print(f"Q5 ✅  conjugate symmetry of a real signal: X[k]=X[N-k]*, so only X[0..N/2] needs computing (half the work saved)")
print("\n🎉 DFT whiteboard challenge passed! X[k]=Σx[n]e^(-2πikn/N) is now second nature.")

In [ ]:
# ✏️ Self-review for this lesson
l37_review = {
    "dft_formula":            None,  # remember X[k]=Σx[n]e^{-2πikn/N}? True/False
    "naive_dft_implemented":  None,  # naive_dft implemented and passes the allclose check? True/False
    "physical_frequency":     None,  # understand f_k = k·sr/N? True/False
    "on2_complexity":         None,  # understand the O(N²) double loop, and know the FFT is O(NlogN)? True/False
    "whiteboard_passed":      None,  # whiteboard challenge done on paper? True/False
    "l21_matrix_bridge":      None,  # can say that L21's W @ x is equivalent to this lesson's Σ formula? True/False
}

unfilled = [k for k, v in l37_review.items() if v is None]
assert not unfilled, f'not filled in yet: {unfilled}'
weak = [k for k, v in l37_review.items() if v is False]
if weak:
    print(f'⚠️  needs more work: {weak}')
else:
    print('✅ L37 fully cleared! On to L38: folding the repeated DFT work into butterflies')

---

→ **Next lesson** · [L38 · The FFT butterfly](L38_fft_butterfly.ipynb)

> Next lesson covers **the FFT butterfly**: the even/odd split, the butterfly `E[k] + W^k·O[k]`, from O(N²) to O(N log N).